# Housing Price Prediction using Linear Regression

**Course:** Introduction to Machine Learning / Data Science  
**Dataset:** Housing.csv — 545 records, 13 features  
**Model:** Linear Regression  
**Objective:** Predict house prices from structural and amenity-based features

---

**Group Members**

| Name | Student ID |
|---|---|
| Nimrat Singh | 2423361 |
| Mayur Garg | 2423358 |
| Anay Mittal | 2423357 |
| Chetanye Gulati | 2423365 |

## Table of Contents

1. [Research Question, Objectives and Hypotheses](#section1)  
2. [Importing Libraries](#section2)  
3. [Data Loading](#section3)  
4. [Data Description](#section4)  
5. [Exploratory Data Analysis](#section5)  
   - 5.1 Missing Values  
   - 5.2 Data Types and Wrangling  
   - 5.3 Visualisations  
6. [Train / Test Split and Feature Scaling](#section6)  
7. [Model — Linear Regression](#section7)  
   - 7.1 Model Choice Reasoning  
   - 7.2 Model Architecture  
   - 7.3 Model Training  
8. [Results — Testing and Evaluation](#section8)  
   - 8.1 Error Metrics  
   - 8.2 Diagram of Line Fit  
9. [Conclusions, Statistics and Hypothesis Outcome](#section9)

---
<a id='section1'></a>
## 1. Research Question, Objectives and Hypotheses

### Research Question

Can we accurately predict the price of a house based on its physical characteristics and available amenities?

### Objectives

1. Load and examine the Housing dataset to understand its structure, dimensions and data types.
2. Perform exploratory data analysis to uncover patterns, distributions and relationships between features and house price.
3. Prepare the data for modelling by encoding categorical variables and scaling numerical features.
4. Train a Linear Regression model to predict house prices.
5. Evaluate the model using standard regression metrics and interpret the results.

### Hypotheses

**Null Hypothesis (H0):**  
The physical features of a house such as area, number of bedrooms, bathrooms and available amenities have no significant linear relationship with its price.

**Alternative Hypothesis (H1):**  
The physical features of a house such as area, number of bedrooms, bathrooms and available amenities have a significant linear relationship with its price.

The hypothesis will be evaluated at the end of the project using the R-squared score obtained from the trained model.

---
<a id='section2'></a>
## 2. Importing Libraries

The following libraries are used throughout this project:

- **pandas** — data loading, manipulation and analysis
- **numpy** — numerical operations
- **matplotlib and seaborn** — data visualisation
- **scikit-learn** — machine learning utilities including preprocessing, model training and evaluation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

# Set a consistent plot style
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')

print("All libraries imported successfully.")

---
<a id='section3'></a>
## 3. Data Loading

The dataset is loaded from a CSV file. The first step is to confirm the file loads correctly, check its dimensions and preview the first few records to get a sense of the data.

In [ ]:
df = pd.read_csv('Housing.csv')

print("Dataset loaded successfully.")
print(f"Shape: {df.shape[0]} rows and {df.shape[1]} columns")

In [ ]:
# Preview the first 10 rows
df.head(10)

---
<a id='section4'></a>
## 4. Data Description

This section examines the data types of each column, the statistical summary for numerical features and the unique categories present in categorical columns. This gives us a complete picture of what we are working with before any preprocessing takes place.

In [ ]:
# Data types for all columns
print("Column Names and Data Types")
print("-" * 40)
print(df.dtypes)

In [ ]:
# Statistical summary of numerical columns
print("Statistical Summary of Numerical Features")
df.describe()

In [ ]:
# Unique values in categorical columns
categorical_cols = ['mainroad', 'guestroom', 'basement',
                    'hotwaterheating', 'airconditioning',
                    'prefarea', 'furnishingstatus']

print("Unique Values in Categorical Columns")
print("-" * 40)
for col in categorical_cols:
    print(f"{col:20s}: {df[col].unique().tolist()}")

In [ ]:
# Count of each furnishing category
print("Furnishing Status Distribution")
print(df['furnishingstatus'].value_counts())

---
<a id='section5'></a>
## 5. Exploratory Data Analysis

Exploratory Data Analysis (EDA) helps us understand the distribution of the data, check for quality issues and identify patterns before modelling. This section is divided into three parts: checking for missing values, correcting data types through wrangling, and producing visualisations.

### 5.1 Missing Values

Before doing anything else, we check whether the dataset contains any missing values. Missing data can distort model training and must be addressed.

In [ ]:
missing = df.isnull().sum()
print("Missing Values per Column")
print("-" * 40)
print(missing)
print()
print(f"Total missing values across entire dataset: {missing.sum()}")

The dataset contains no missing values across any of its 13 columns. This means no imputation or row removal is required, and we can proceed directly to encoding.

### 5.2 Data Types and Wrangling

Several columns contain text-based yes/no values and a multi-class furnishing status column. Machine learning models require numerical input, so these columns need to be encoded before they can be used.

The binary yes/no columns are mapped to 1 and 0 respectively. The furnishing status column is ordinally encoded — furnished receives the highest value (2), semi-furnished receives 1, and unfurnished receives 0 — which reflects a natural ordering in quality and price expectation.

In [ ]:
# Keep a copy of the original for reference
df_original = df.copy()

# Encode binary yes/no columns
binary_cols = ['mainroad', 'guestroom', 'basement',
               'hotwaterheating', 'airconditioning', 'prefarea']

for col in binary_cols:
    df[col] = df[col].map({'yes': 1, 'no': 0})

# Ordinal encoding for furnishing status
furnish_map = {'furnished': 2, 'semi-furnished': 1, 'unfurnished': 0}
df['furnishingstatus'] = df['furnishingstatus'].map(furnish_map)

print("Encoding complete.")
print()
print("Updated Data Types")
print("-" * 40)
print(df.dtypes)

In [ ]:
# Confirm the encoding looks correct
print("First 5 rows after encoding")
df.head()

### 5.3 Visualisations

The following plots are used to understand the distribution of house prices, the relationships between key features and price, and the correlations among all variables in the dataset.

In [ ]:
# Figure 1: Price Distribution
fig, ax = plt.subplots(figsize=(9, 4))

ax.hist(df['price'], bins=35, color='steelblue', edgecolor='white', linewidth=0.6)
ax.set_title('Distribution of House Prices', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Price', fontsize=11)
ax.set_ylabel('Number of Houses', fontsize=11)
ax.axvline(df['price'].mean(), color='crimson', linestyle='--', linewidth=1.5, label=f"Mean: {df['price'].mean():,.0f}")
ax.axvline(df['price'].median(), color='darkorange', linestyle='--', linewidth=1.5, label=f"Median: {df['price'].median():,.0f}")
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"Mean price  : {df['price'].mean():,.0f}")
print(f"Median price: {df['price'].median():,.0f}")
print(f"Std dev     : {df['price'].std():,.0f}")

The price distribution is right-skewed, meaning most houses are priced at the lower to mid range while a smaller number of high-value properties pull the mean above the median. This is a common pattern in housing datasets.

In [ ]:
# Figure 2: Area vs Price
fig, ax = plt.subplots(figsize=(9, 5))

scatter = ax.scatter(df['area'], df['price'], alpha=0.55, c=df['price'],
                     cmap='coolwarm', edgecolors='none', s=50)
plt.colorbar(scatter, ax=ax, label='Price')
ax.set_title('Area vs House Price', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Area (sq ft)', fontsize=11)
ax.set_ylabel('Price', fontsize=11)

plt.tight_layout()
plt.show()

There is a clear positive relationship between area and price — larger houses tend to cost more. This confirms that area will likely be one of the strongest predictors in the regression model.

In [ ]:
# Figure 3: Average Price by Number of Bedrooms and Bathrooms
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bedroom_avg = df.groupby('bedrooms')['price'].mean()
axes[0].bar(bedroom_avg.index, bedroom_avg.values, color='steelblue', edgecolor='white', linewidth=0.7)
axes[0].set_title('Average Price by Number of Bedrooms', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Bedrooms', fontsize=11)
axes[0].set_ylabel('Average Price', fontsize=11)
for i, v in enumerate(bedroom_avg.values):
    axes[0].text(bedroom_avg.index[i], v + 30000, f'{v/1e6:.1f}M', ha='center', fontsize=9)

bathroom_avg = df.groupby('bathrooms')['price'].mean()
axes[1].bar(bathroom_avg.index, bathroom_avg.values, color='coral', edgecolor='white', linewidth=0.7)
axes[1].set_title('Average Price by Number of Bathrooms', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Bathrooms', fontsize=11)
axes[1].set_ylabel('Average Price', fontsize=11)
for i, v in enumerate(bathroom_avg.values):
    axes[1].text(bathroom_avg.index[i], v + 30000, f'{v/1e6:.1f}M', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Figure 4: Average Price by Stories and Furnishing Status
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

story_avg = df.groupby('stories')['price'].mean()
axes[0].bar(story_avg.index, story_avg.values, color='mediumseagreen', edgecolor='white', linewidth=0.7)
axes[0].set_title('Average Price by Number of Stories', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Stories', fontsize=11)
axes[0].set_ylabel('Average Price', fontsize=11)

furnish_avg = df.groupby('furnishingstatus')['price'].mean()
furnish_labels = {0: 'Unfurnished', 1: 'Semi-Furnished', 2: 'Furnished'}
x_labels = [furnish_labels[i] for i in furnish_avg.index]
axes[1].bar(x_labels, furnish_avg.values, color=['#d9534f', '#f0ad4e', '#5cb85c'], edgecolor='white', linewidth=0.7)
axes[1].set_title('Average Price by Furnishing Status', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Furnishing Status', fontsize=11)
axes[1].set_ylabel('Average Price', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Figure 5: Correlation Heatmap
fig, ax = plt.subplots(figsize=(11, 8))

corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, ax=ax, linewidths=0.5, annot_kws={'size': 9},
            vmin=-1, vmax=1, center=0, square=True)

ax.set_title('Correlation Matrix — All Features', fontsize=13, fontweight='bold', pad=14)
plt.tight_layout()
plt.show()

From the correlation heatmap, the features most strongly correlated with price are area, bathrooms, air conditioning, stories and preferred area. Notably, bedrooms shows a weaker correlation than one might expect, possibly because larger homes with more bedrooms also tend to have other premium features that absorb the price effect.

In [ ]:
# Figure 6: Amenity Features vs Average Price (Binary columns)
amenity_cols = ['mainroad', 'guestroom', 'basement',
                'hotwaterheating', 'airconditioning', 'prefarea']
amenity_labels = ['Main Road', 'Guest Room', 'Basement',
                  'Hot Water Heating', 'Air Conditioning', 'Preferred Area']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, (col, label) in enumerate(zip(amenity_cols, amenity_labels)):
    avg = df.groupby(col)['price'].mean()
    axes[i].bar(['No', 'Yes'], avg.values, color=['#6baed6', '#2171b5'], edgecolor='white')
    axes[i].set_title(label, fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Average Price', fontsize=9)
    for j, v in enumerate(avg.values):
        axes[i].text(j, v + 20000, f'{v/1e6:.2f}M', ha='center', fontsize=9)

fig.suptitle('Average House Price by Amenity Availability', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

All six amenity features show a clear positive effect on average price. Air conditioning and preferred area produce the largest price differences between houses that have them and those that do not, making them valuable predictors for the model.

---
<a id='section6'></a>
## 6. Train / Test Split and Feature Scaling

### Feature and Target Separation

The target variable is the house price (the column we want to predict). All remaining columns become input features for the model.

In [ ]:
X = df.drop('price', axis=1)
y = df['price']

print(f"Features (X) shape : {X.shape}")
print(f"Target   (y) shape : {y.shape}")
print()
print("Feature columns used for training:")
for col in X.columns:
    print(f"  - {col}")

### Train / Test Split

The dataset is split into a training set (80%) and a test set (20%). The training set is used to fit the model, while the test set is held back and used only for final evaluation. A fixed random state ensures reproducibility.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size : {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test set size     : {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)")

### Feature Scaling

Linear Regression is sensitive to the scale of input features. A feature like area with values in the thousands would dominate over a binary feature like mainroad with values of 0 or 1. StandardScaler standardises each feature to have a mean of 0 and a standard deviation of 1, ensuring all features contribute fairly to the model.

The scaler is fitted only on the training set to prevent data leakage. The same fitted scaler is then applied to transform the test set.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("StandardScaler applied.")
print()
print("Before scaling — area column stats (training set):")
print(f"  Mean : {X_train['area'].mean():.2f}")
print(f"  Std  : {X_train['area'].std():.2f}")
print()
print("After scaling — area column stats (training set):")
area_idx = list(X.columns).index('area')
print(f"  Mean : {X_train_scaled[:, area_idx].mean():.4f}")
print(f"  Std  : {X_train_scaled[:, area_idx].std():.4f}")

---
<a id='section7'></a>
## 7. Model — Linear Regression

### 7.1 Model Choice Reasoning

Linear Regression was chosen as the model for this project for the following reasons.

The target variable (house price) is a continuous numerical value. This makes the problem a regression task, and Linear Regression is the most direct and interpretable approach for such tasks.

The dataset is relatively small (545 records) and has 12 input features. Complex models like neural networks or gradient boosting would be prone to overfitting on a dataset of this size without extensive tuning. Linear Regression is better suited here.

Linear Regression produces a set of coefficients — one for each feature — that directly tell us how much each feature contributes to the price. This interpretability is valuable for drawing conclusions and testing our hypothesis.

Finally, it serves as a strong and well-understood baseline. If a more complex model were to be explored later, its performance would be compared against this baseline.

The mathematical equation fitted by the model is:

    price = w1*area + w2*bedrooms + w3*bathrooms + ... + w12*furnishingstatus + b

where each w is a learned coefficient and b is the bias (intercept) term.

### 7.2 Model Architecture

The pipeline followed for this project is shown below.

    Raw Dataset (545 rows, 13 columns)
            |
            v
    Data Wrangling
      - Binary encoding of yes/no columns (0/1)
      - Ordinal encoding of furnishingstatus (0, 1, 2)
            |
            v
    Feature / Target Split
      X : 12 input features
      y : price (target)
            |
            v
    Train/Test Split (80% / 20%)
      Training set : 436 samples
      Test set     : 109 samples
            |
            v
    StandardScaler
      Applied to X_train (fit + transform)
      Applied to X_test  (transform only)
            |
            v
    Linear Regression Model
      Fitted on X_train_scaled, y_train
            |
            v
    Predictions on X_test_scaled
            |
            v
    Evaluation
      MAE, RMSE, R-squared

### 7.3 Model Training

In [ ]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

print("Model trained successfully.")
print()
print(f"Intercept (bias term) : {model.intercept_:,.2f}")

In [ ]:
# Display the learned coefficients for each feature
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
}).sort_values('Coefficient', ascending=False).reset_index(drop=True)

print("Feature Coefficients (sorted by impact on price)")
print("-" * 45)
print(coef_df.to_string(index=False))

In [ ]:
# Visualise feature coefficients
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['steelblue' if c >= 0 else 'crimson' for c in coef_df['Coefficient']]
bars = ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white', linewidth=0.6)

ax.set_title('Feature Coefficients — Linear Regression Model', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Coefficient Value', fontsize=11)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')

for bar, val in zip(bars, coef_df['Coefficient']):
    ax.text(val + 5000, bar.get_y() + bar.get_height()/2,
            f'{val:,.0f}', va='center', fontsize=8.5)

plt.tight_layout()
plt.show()

The coefficient chart confirms that bathrooms, area and air conditioning are the three most influential predictors of house price. Each additional unit increase in these features (after scaling) corresponds to the largest increase in predicted price. Bedrooms, while intuitively important, has a smaller coefficient, which suggests that its effect on price is partially explained by other correlated features already in the model.

---
<a id='section8'></a>
## 8. Results — Testing and Evaluation

### 8.1 Error Metrics

The trained model is applied to the unseen test set. Three standard regression metrics are computed to assess how accurately the model predicts house prices.

- **MAE (Mean Absolute Error)** — the average absolute difference between predicted and actual prices. Easy to interpret in the same units as price.
- **RMSE (Root Mean Squared Error)** — similar to MAE but penalises large errors more heavily.
- **R-squared (R2 Score)** — the proportion of variance in the target variable explained by the model. A value of 1.0 is a perfect fit; values closer to 0 indicate the model explains little of the variation.

In [ ]:
y_pred = model.predict(X_test_scaled)

mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print("Model Evaluation on Test Set")
print("=" * 45)
print(f"  Mean Absolute Error  (MAE)  :  {mae:>15,.2f}")
print(f"  Mean Squared Error   (MSE)  :  {mse:>15,.2f}")
print(f"  Root Mean Sq Error   (RMSE) :  {rmse:>15,.2f}")
print(f"  R-squared Score      (R2)   :  {r2:>15.4f}")
print()
print(f"  The model explains {r2*100:.1f}% of the variance in house prices.")

### 8.2 Diagram of Line Fit and Residuals

Two diagnostic plots are produced. The first compares actual prices against predicted prices — a perfect model would show all points along the red dashed line. The second is a residuals plot, which checks whether prediction errors are randomly distributed around zero.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Model Performance on Test Set', fontsize=14, fontweight='bold', y=1.01)

# Plot 1: Actual vs Predicted
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())

axes[0].scatter(y_test, y_pred, alpha=0.6, color='steelblue', edgecolors='none', s=55)
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect prediction')
axes[0].set_title('Actual vs Predicted Prices\n(Diagram of Line Fit)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Actual Price', fontsize=11)
axes[0].set_ylabel('Predicted Price', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].text(0.05, 0.93, f'R² = {r2:.4f}', transform=axes[0].transAxes,
             fontsize=11, color='darkred', fontweight='bold')

# Plot 2: Residuals
residuals = y_test.values - y_pred

axes[1].scatter(y_pred, residuals, alpha=0.6, color='coral', edgecolors='none', s=55)
axes[1].axhline(y=0, color='black', linewidth=1.5, linestyle='--')
axes[1].set_title('Residuals Plot\n(Prediction Errors Distribution)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted Price', fontsize=11)
axes[1].set_ylabel('Residual (Actual - Predicted)', fontsize=11)
axes[1].text(0.05, 0.93, f'Mean residual = {residuals.mean():,.0f}', transform=axes[1].transAxes,
             fontsize=10, color='darkred')

plt.tight_layout()
plt.show()

The actual vs predicted plot shows a reasonable alignment with the diagonal line of perfect prediction, confirming the model has learned a useful relationship. There is, however, some spread around the line, which indicates the model has limitations — particularly for very high-priced properties.

The residuals plot shows that errors are distributed reasonably close to zero, with no strong systematic pattern. A slight tendency toward larger residuals at higher predicted prices suggests the model slightly underestimates expensive houses, which is consistent with the right-skewed distribution of prices noted in the EDA.

In [ ]:
# Detailed prediction comparison — first 15 test samples
comparison_df = pd.DataFrame({
    'Actual Price': y_test.values[:15],
    'Predicted Price': y_pred[:15].round(0),
    'Difference': (y_test.values[:15] - y_pred[:15]).round(0)
})
comparison_df.index = range(1, 16)
comparison_df.index.name = 'Sample'
print("Actual vs Predicted — First 15 Test Samples")
print(comparison_df.to_string())

---
<a id='section9'></a>
## 9. Conclusions, Statistics and Hypothesis Outcome

### Summary of Results

In [ ]:
print("Project Summary")
print("=" * 55)
print(f"  Dataset         : Housing.csv")
print(f"  Total records   : 545")
print(f"  Features used   : 12")
print(f"  Target variable : Price")
print(f"  Model           : Linear Regression")
print()
print("  Data Split")
print(f"    Training set  : 436 samples (80%)")
print(f"    Test set      : 109 samples (20%)")
print()
print("  Model Performance on Test Set")
print(f"    MAE           : {mae:,.2f}")
print(f"    RMSE          : {rmse:,.2f}")
print(f"    R-squared     : {r2:.4f}  ({r2*100:.1f}% variance explained)")
print()
print("  Top Predictors (by coefficient magnitude)")
for _, row in coef_df.head(5).iterrows():
    print(f"    {row['Feature']:20s}: {row['Coefficient']:>12,.2f}")

### Hypothesis Outcome

The model achieved an R-squared score of approximately 0.65, meaning it explains around 65% of the variance in house prices using the 12 input features.

On the basis of this result, **the Null Hypothesis (H0) is rejected**. The physical features and amenities of a house do have a significant and measurable linear relationship with its price. Features such as area, number of bathrooms, air conditioning, number of stories and preferred area location all contribute meaningfully to the predicted price.

### Key Findings from EDA

The distribution of house prices in the dataset is right-skewed, with a majority of properties priced between 3 million and 7 million and a smaller number of premium properties above 10 million.

Area is the single most intuitive predictor — the scatter plot clearly shows that price increases with area. Air conditioning, preferred area location and furnishing status each add a substantial premium, indicating that buyers place significant value on comfort and convenience features.

### Limitations and Future Work

While the model performs reasonably well, an R-squared of 0.65 means approximately 35% of the price variation remains unexplained. This is expected given that Linear Regression assumes a strictly linear relationship between features and price, which may not always hold.

The following improvements could be explored in future work:

- Applying log transformation to the price variable to reduce the effect of skewness and potentially improve model fit.
- Testing non-linear models such as Decision Tree Regression, Random Forest or Gradient Boosting, which can capture interactions between features.
- Using cross-validation instead of a single train/test split for a more reliable performance estimate.
- Exploring polynomial features to capture non-linear relationships within the linear regression framework.

### Conclusion

This project demonstrated the end-to-end workflow of a supervised machine learning project — from data loading and exploratory analysis, through preprocessing and model training, to evaluation and interpretation. Linear Regression provided a solid and interpretable baseline for predicting house prices from structural and amenity-based features, and the results support the hypothesis that these features are meaningful predictors of price.